# Constructing and deconstructing beams using modal representations

The transverse intensity profile of any laser beam can be represented as a summation over infinitely many modes in a basis set. In this tutorial, we consider two different basis sets: the Hermite-Gaussians and the Laguerre-Gaussians. Other sets, such as the Zernike polynomials, are also possible but are not currently implemented in LASY. In this tutorial, beams are constructed using a complex dictionary of the modal coefficients. It is shown that a decomposition of these beams back into the basis sets returns the original mode coefficients, provided the assumed beam waist remains the same. First, we import the required modules:

In [ ]:
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c, epsilon_0

from lasy.laser import Laser
from lasy.profiles import CombinedLongitudinalTransverseProfile
from lasy.profiles.longitudinal import ContinuousWaveProfile
from lasy.profiles.transverse import (
    GaussianTransverseProfile,
)
from lasy.utils.mode_decomposition import (
    hermite_gauss_composition,
    hermite_gauss_decomposition,
    laguerre_gauss_composition,
    laguerre_gauss_decomposition,
)

Then we initialise the laser. We will just use a CW laser, since we only care about the spatial profile.

In [ ]:
peak_fluence = 1.0e4  # J/m^2
spot_size = 10e-3
wavelength = 800e-9
omega0 = 2 * np.pi * c / wavelength
pol = (1, 0)

dimensions = "xyt"  # Use cylindrical geometry
lo = (-2.0 * spot_size, -2.0 * spot_size, None)  # Lower bounds of the simulation box
hi = (2.0 * spot_size, 2.0 * spot_size, None)  # Upper bounds of the simulation box
num_points = (256, 256, 1)  # Number of points in each dimension

long_prof = ContinuousWaveProfile(wavelength)
tran_prof = GaussianTransverseProfile(spot_size)
laser_profile = CombinedLongitudinalTransverseProfile(
    wavelength, pol, long_prof, tran_prof, peak_fluence=peak_fluence
)

laser = Laser(dimensions, lo, hi, num_points, laser_profile)

# Plot the xt laser profile
laser.show(envelope_type="intensity")  # In this case this is the fluence (CW beam)

# Plot the transverse laser profile
intensity = epsilon_0 * c / 2 * np.abs(laser.grid.get_temporal_field()) ** 2
fluence = np.sum(intensity, axis=-1).T * laser.grid.dx[-1]
extent = (
    laser.grid.axes[0].min() * 1e3,
    laser.grid.axes[0].max() * 1e3,
    laser.grid.axes[1].min() * 1e3,
    laser.grid.axes[1].max() * 1e3,
)
plt.figure()
plt.imshow(fluence, extent=extent, cmap="Reds")
plt.xlabel("x (mm)")
plt.ylabel("y (mm)")
plt.colorbar(label=r"Fluence (J/m$^2$)")

We now define some random coefficients. These can be used to construct arbitrary beams as superpositions of Hermite- and Laguerre-Gaussian modes (see Denoised Laser tutorial for an example of expressing an experimental laser profile as a summation of Hermite-Gaussian modes). 

In [ ]:
# Modal coefficients used to construct the beam
modes = {}
modes[(0, 0)] = 1 - 0.2j
modes[(0, 1)] = -0.4 - 0.2j
modes[(1, 0)] = 0.5 + 1j
modes[(2, 2)] = -0.05 + 0.1j

We can use these mode coefficients to generate laser pulses. Here is shown first for the Hermite-Gaussian modes...

In [ ]:
HG_laser = deepcopy(laser)  # Make a copy of the input grid
hermite_gauss_composition(
    HG_laser, spot_size, spot_size, modes, skipAsymmetricModes=False
)

HG_laser.show(envelope_type="intensity")  # In this case this is the fluence (CW beam)

intensity = epsilon_0 * c / 2 * np.abs(HG_laser.grid.get_temporal_field()) ** 2
fluence = np.sum(intensity, axis=-1).T * HG_laser.grid.dx[-1]
extent = (
    HG_laser.grid.axes[0].min() * 1e3,
    HG_laser.grid.axes[0].max() * 1e3,
    HG_laser.grid.axes[1].min() * 1e3,
    HG_laser.grid.axes[1].max() * 1e3,
)
plt.figure()
plt.imshow(fluence, extent=extent, cmap="Reds")
plt.xlabel("x (mm)")
plt.ylabel("y (mm)")
plt.colorbar(label=r"Fluence (J/m$^2$)")

calculatedModes = hermite_gauss_decomposition(HG_laser, spot_size, spot_size, 3, 3)
print("Mode coefficients for HG beam (up to third order)")
for calcModeKey in calculatedModes.keys():
    print(
        "%i,%i :  %.2f  ,  %.2f i"
        % (
            calcModeKey[0],
            calcModeKey[1],
            np.real(calculatedModes[calcModeKey]),
            np.imag(calculatedModes[calcModeKey]),
        )
    )

... where we have also confirmed that the recovered mode coefficients match the input mode coefficients. Similarly, we could use a Laguerre-Gaussian modal basis (which is particularly relevant for beams with orbital angular momentum):

In [ ]:
LG_laser = deepcopy(laser)  # Make a copy of the input grid
laguerre_gauss_composition(LG_laser, spot_size, modes, skipAsymmetricModes=False)

LG_laser.show(envelope_type="intensity")  # In this case this is the fluence (CW beam)

intensity = epsilon_0 * c / 2 * np.abs(LG_laser.grid.get_temporal_field()) ** 2
fluence = np.sum(intensity, axis=-1).T * LG_laser.grid.dx[-1]
extent = (
    LG_laser.grid.axes[0].min() * 1e3,
    LG_laser.grid.axes[0].max() * 1e3,
    LG_laser.grid.axes[1].min() * 1e3,
    LG_laser.grid.axes[1].max() * 1e3,
)
plt.figure()
plt.imshow(fluence, extent=extent, cmap="Reds")
plt.xlabel("x (mm)")
plt.ylabel("y (mm)")
plt.colorbar(label=r"Fluence (J/m$^2$)")

calculatedModes = laguerre_gauss_decomposition(LG_laser, spot_size, 3, 3)
print("Mode coefficients for HG beam (up to third order)")
for calcModeKey in calculatedModes.keys():
    print(
        "%i,%i :  %.2f  ,  %.2f i"
        % (
            calcModeKey[0],
            calcModeKey[1],
            np.real(calculatedModes[calcModeKey]),
            np.imag(calculatedModes[calcModeKey]),
        )
    )